In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.\
            builder.\
                appName('Window Fucntion').\
                    getOrCreate()

In [0]:
emp_schema = "id INT, name STRING, department STRING, salary DOUBLE, hire_date STRING, country STRING, age INT, gender STRING"

emp_data = [
    (1, "John Smith", "Engineering", 85000.0, "2021-03-15", "USA", 30, "Male"),
    (2, "Alice Johnson", "Engineering", 92000.0, "2020-07-01", "USA", 35, "Female"),
    (3, "Bob Williams", "Sales", 65000.0, "2022-01-20", "UK", 28, None),
    (4, "Carol Brown", "Sales", 68000.0, "2019-11-05", "UK", 40, "Female"),
    (5, "David Jones", "HR", 55000.0, "2021-09-10", "Canada", 32, "Male"),
    (6, "Eva Garcia", "HR", 58000.0, "2023-02-28", "Canada", 26, "Female"),
    (7, "Frank Miller", "Engineering", 78000.0, "2018-05-22", "Germany", 45, "Male"),
    (8, "Grace Davis", "Marketing", 72000.0, "2022-08-14", "Germany", 29, "Female"),
    (9, "Henry Wilson", "Marketing", 70000.0, "2020-12-01", "India", 33, "Other"),
    (10, "Ivy Taylor", "Finance", 95000.0, "2017-06-30", "India", 50, "Female"),
    (11, "Jack Anderson", "Finance", 89000.0, "2021-04-18", "Australia", 38, "Male"),
    (12, "Karen Thomas", "Engineering", 81000.0, "2023-03-07", "Australia", 25, "Female"),
    (13, "Leo Moore", "Sales", 62000.0, "2019-10-25", "USA", 37, None),
    (14, "Mia Martin", "Engineering", 90000.0, "2022-05-11", "USA", 27, "Female"),
    (15, "Nick Lee", "HR", 60000.0, "2018-02-09", "UK", 42, "Male"),
    (16, "Olivia Harris", "Marketing", 74000.0, "2020-09-03", "Canada", 31, "Female"),
    (17, "Paul Clark", "Finance", 87000.0, "2023-07-19", "Germany", 24, "Male"),
    (18, "Quinn Lewis", "Engineering", 83000.0, "2021-12-12", "India", 34, "Other"),
    (19, "Rachel Walker", "Sales", 66000.0, "2019-08-08", "Australia", 36, "Female"),
    (20, "Sam Hall", "Marketing", 71000.0, "2022-11-27", "USA", 23, None),
]

## Creating the dataFrame
emp = spark.createDataFrame(
            data = emp_data,
            schema = emp_schema
)

In [0]:
# get unique data/row
unique_data = emp.distinct()
unique_data.show()

In [0]:
# distinct department
emp_dept = emp.select("department").\
                    distinct()
emp_dept.show()

#### Window Function
**A Window Function is a function that performs calculations across a set of related rows without combining those rows into a single row.**

* In simple terms:
    - `GROUPBY` combines multiple rows into one row per group.
    - `Window Functions` perform calculations across multiple rows but retain the original rows.

In [0]:
## Window Fucntion
# max salary in each dept
from pyspark.sql.window import Window
from pyspark.sql.functions import max, col, desc

window_specs = Window.partitionBy("department")
max_func = max(col("salary")).over(window_specs)

emp1= emp.withColumn("max_salary", max_func)
display(emp1)

In [0]:
# "group by" that keeps ALL rows (no window function):
# aggregate with groupBy, then join back to the original rows
from pyspark.sql.functions import max, col, desc

dept_max = emp.groupBy("department") \
              .agg(max("salary").alias("dept_max_salary"))

emp_all_rows = emp.join(dept_max, on="department", how="left") \
                 .orderBy("department", desc("salary"))

display(dept_max)
display(emp_all_rows)

In [0]:
group_by_demo = emp.groupBy('department').\
                    agg(max("salary").alias("max_salary"))

display(group_by_demo)

In [0]:
# assign the salary with their departmwnt and label the rank
from pyspark.sql.window import Window
from pyspark.sql.functions import max, col, desc, row_number

window_specs = Window.partitionBy("department").orderBy(desc("salary"))
rank_func = row_number().over(window_specs)

emp2 = emp.withColumn("rank", rank_func)
display(emp2)

In [0]:
# find second highest salary in each department
from pyspark.sql.window import Window
from pyspark.sql.functions import max, col, desc, dense_rank

window_specs = Window.partitionBy("department").\
                    orderBy(desc("salary"))

rank_func = dense_rank().\
                    over(window_specs)

emp2 = emp.withColumn("rank", rank_func).\
                where("rank = 2")
display(emp2)


In [0]:
## Alternative
# highrst salary in each dept
emp_alt = emp.withColumn("max_salary", max("salary").over(Window.partitionBy("department")))
emp_alt.show()

In [0]:
# seconf highest salary in each dept
from pyspark.sql.functions import expr, row_number

emp_sec = emp.withColumn("rn", expr("row_number() over (partition by department order by salary desc)")).\
                        where("rn = 2")
emp_sec.show()